In [1]:
import pandas as pd
import os
import sys

In [2]:
# Get project root
project_root = os.path.abspath("..")

# Add to Python path
sys.path.append(project_root)

print(project_root)

/home/chaithanaya/Documents/ai-route-optimizer


In [3]:
from services.route_optimizer import (
    nearest_neighbor_optimizer
)

In [10]:
trips_path = os.path.join(
    project_root,
    "data",
    "historical_trips.csv"
)

cache_path = os.path.join(
    project_root,
    "data",
    "cleaned_route_cache.csv"
)

trips_df = pd.read_csv(trips_path)

cache_df = pd.read_csv(cache_path)

In [11]:
grouped_trips = trips_df.groupby(
    "trip_id"
)

In [12]:
# Get first 10 unique trip IDs
trip_ids_10 = trips_df["trip_id"].unique()[:10]

# Filter dataframe
trips_df_10 = trips_df[
    trips_df["trip_id"].isin(trip_ids_10)
]

# Group only these 10 trips
grouped_trips_10 = trips_df_10.groupby("trip_id")

In [13]:
def build_duration_matrix(
    location_ids,
    cache_df
):

    n = len(location_ids)

    matrix = [

        [0 for _ in range(n)]

        for _ in range(n)
    ]

    for i in range(n):

        for j in range(n):

            if i == j:
                continue

            origin = location_ids[i]

            destination = location_ids[j]

            match = cache_df[

                (cache_df["origin_id"] == origin) &

                (cache_df["destination_id"] == destination)
            ]

            if len(match) > 0:

                duration = match.iloc[0][
                    "duration_minutes"
                ]

                matrix[i][j] = duration

            else:

                # Large penalty if route missing
                matrix[i][j] = 9999

    return matrix

In [14]:
optimized_rows = []

for trip_id, trip_data in grouped_trips_10:

    print(f"\nOptimizing Trip: {trip_id}")

    # -----------------------------------
    # SORT ORIGINAL ROUTE
    # -----------------------------------

    trip_data = trip_data.sort_values(
        by="stop_number"
    )

    # -----------------------------------
    # EXTRACT IDS/NAMES
    # -----------------------------------

    location_ids = list(
        trip_data["location_id"]
    )

    store_names = list(
        trip_data["store_name"]
    )

    # -----------------------------------
    # BUILD MATRIX
    # -----------------------------------

    duration_matrix = build_duration_matrix(
        location_ids,
        cache_df
    )

    # -----------------------------------
    # OPTIMIZE
    # -----------------------------------

    result = nearest_neighbor_optimizer(
        duration_matrix
    )

    optimized_indexes = result[
        "optimized_route"
    ]

    optimized_route = [

        store_names[i]

        for i in optimized_indexes
    ]

    optimized_route_string = (
        " → ".join(optimized_route)
    )

    optimized_rows.append({

        "trip_id": trip_id,

        "optimized_route":
            optimized_route_string,

        "optimized_duration_min":
            result[
                "total_duration_min"
            ]
    })


Optimizing Trip: 1

Optimizing Trip: 2

Optimizing Trip: 3

Optimizing Trip: 4

Optimizing Trip: 5

Optimizing Trip: 6

Optimizing Trip: 7

Optimizing Trip: 8

Optimizing Trip: 9

Optimizing Trip: 10


In [15]:
optimized_df = pd.DataFrame(
    optimized_rows
)

output_path = os.path.join(
    project_root,
    "data/demo",
    "optimized_routes_demo.csv"
)

optimized_df.to_csv(
    output_path,
    index=False
)

print("\nOptimization Complete!")

print(optimized_df.head())


Optimization Complete!
   trip_id                                    optimized_route  \
0        1  Ratnadeep Supermarket → Cafe Niloufer → Apollo...   
1        2  Decathlon Madhapur → Joyalukkas → South India ...   
2        3  Karachi Bakery → Naturals Salon → Bata Shoe St...   
3        4  Karachi Bakery → Inorbit Mall Shoppers Stop → ...   
4        5  Reliance Smart Jubilee → Joyalukkas → Bajaj El...   

   optimized_duration_min  
0                  281.05  
1                   77.51  
2                  206.78  
3                  211.56  
4                  129.54  
